In [2]:
import pymatgen
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar
from pymatgen.io.cif import CifWriter
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import Slab, SlabGenerator, generate_all_slabs, Structure, Lattice, ReconstructionGenerator
from matplotlib import pyplot as plt
import os
from ase import Atoms
from ase.io import read, write
from ase.visualize import view
from ase.build import surface
import numpy as np
import subprocess

In [3]:
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType

def get_site_dos(filename:str, site_index: int, spin = None, erange =(-20,20), orbital: str = 'd'):
    vasprun = Vasprun(filename, parse_projected_eigen=True)
    dos = vasprun.complete_dos
    orb_map = {"s": OrbitalType.s, "p": OrbitalType.p, "d": OrbitalType.d, "f": OrbitalType.f}
    if orbital not in orb_map:
        raise ValueError(f"Unsupported orbital: {orbital}")

    orb = orb_map[orbital]
    return dos.get_band_center(band = orb, sites = [dos.structure[site_index]], erange=erange, spin =spin)

sites_dos = [49,44,69,64,54,59,74,79,9,4,29,24,14,19,34,39]
for site in sites_dos:
    site_dos = get_site_dos(filename="/data2/home/luodh/work/DICP/slab-Pd/111/dos/vasprun.xml", site_index=site, orbital='d')
    print(site, site_dos)  ### results as -1.318744285285996

# site_dos = get_site_dos(filename="/data2/home/luodh/work/DICP/new-particle/19/dos/vasprun.xml", site_index=8, orbital='d')
# print(site_dos)  ##results: -0.46560920176182485 for site 14

49 -1.4408234228293217
44 -1.4409429788558221
69 -1.4405901286248146
64 -1.4414611069632666
54 -1.4401531885073175
59 -1.4412221859880219
74 -1.4407806004720691
79 -1.4405954477422243
9 -1.4406869446491455
4 -1.4411801981754018
29 -1.4409670444243128
24 -1.440617957660211
14 -1.4407695064862462
19 -1.44090139002023
34 -1.4411472582371918
39 -1.441035662806095


## 批量读取位点数据-d带数据

In [16]:
import pip
import pymatgen
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar
from pymatgen.io.cif import CifWriter
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import Slab, SlabGenerator, generate_all_slabs, Structure, Lattice, ReconstructionGenerator
import os
# pip.main(['install','ase'])
from ase import Atoms
from ase.io import read, write
from ase.visualize import view
from ase.build import surface
import numpy as np
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType
# 文件夹路径和参数初始化
folder_path = "rest-MO2/3d/slab/clean/DOS/"
all_folders = os.listdir(folder_path)

# 用于存储结果的字典，按文件夹和参数存储
results = {folder: {"overall": {}, "sites": {}} for folder in all_folders}

# 处理每个文件夹的 dos 数据
for folder in all_folders:
    data_path_vasprun = f"{folder_path}/{folder}/vasprun.xml"
    vasprun = Vasprun(data_path_vasprun)
    complete_dos = vasprun.complete_dos  # 获取完整的 DOS 类
    A = folder[:-2]
    elements_from_bulk = folder[:-2] # 元素
    sites_to_analyze = [3,11,15,19,23]  # 位点
    print(f"{elements_from_bulk}")
    # # 整体值计算
    # results[folder]["overall"]["d_band_center_up"] = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.up, erange=[-20, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_center_down"] = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.down, erange=[-20, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_filling_up"] = complete_dos.get_band_filling(elements=elements_from_bulk, spin=Spin.up, band=OrbitalType.d)
    # results[folder]["overall"]["d_band_filling_down"] = complete_dos.get_band_filling(elements=elements_from_bulk, spin=Spin.down, band=OrbitalType.d)
    # results[folder]["overall"]["d_band_skewness_up"] = complete_dos.get_band_skewness(elements=elements_from_bulk, spin=Spin.up, erange=[-20, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_skewness_down"] = complete_dos.get_band_skewness(elements=elements_from_bulk, spin=Spin.down, erange=[-20, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_kurtosis_up"] = complete_dos.get_band_kurtosis(elements=elements_from_bulk, spin=Spin.up, erange=[-20, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_kurtosis_down"] = complete_dos.get_band_kurtosis(elements=elements_from_bulk, spin=Spin.down, erange=[-20, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_width_up"] = complete_dos.get_band_width(elements=elements_from_bulk, spin=Spin.up, erange=[-20, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_width_down"] = complete_dos.get_band_width(elements=elements_from_bulk, spin=Spin.down, erange=[-20, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_unoccpy_up"] = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.up, erange=[0, 20], band=OrbitalType.d)
    # results[folder]["overall"]["d_band_unoccpy_down"] = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.down, erange=[0, 20], band=OrbitalType.d)

    # 处理每个位点的数据
    for site in sites_to_analyze:
        if site not in results[folder]["sites"]:
            results[folder]["sites"][site] = {}
        
        # 每个位点的计算
        results[folder]["sites"][site]["d_band_center_site_up"] = complete_dos.get_band_center(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[-20, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_center_site_down"] = complete_dos.get_band_center(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[-20, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_filling_site_up"] = complete_dos.get_band_filling(sites=[complete_dos.structure[site]], spin=Spin.up, band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_filling_site_down"] = complete_dos.get_band_filling(sites=[complete_dos.structure[site]], spin=Spin.down, band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_skewness_site_up"] = complete_dos.get_band_skewness(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[-20, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_skewness_site_down"] = complete_dos.get_band_skewness(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[-20, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_kurtosis_site_up"] = complete_dos.get_band_kurtosis(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[-20, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_kurtosis_site_down"] = complete_dos.get_band_kurtosis(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[-20, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_width_site_up"] = complete_dos.get_band_width(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[-20, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_width_site_down"] = complete_dos.get_band_width(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[-20, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_unoccpy_site_up"] = complete_dos.get_band_center(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[0, 20], band=OrbitalType.d)
        results[folder]["sites"][site]["d_band_unoccpy_site_down"] = complete_dos.get_band_center(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[0, 20], band=OrbitalType.d)        
# 按照文件夹进行输出
def print_folder_based_output(results):
    for folder, data in results.items():
        print(f"===== {folder} =====")
        
        # 打印整体数据
        print("Overall Values:")
        for param, value in data["overall"].items():
            print(f"{param}: {value}")
        
        # 打印每个位点的数据
        print("\nSite-specific Values:")
        for site, site_data in data["sites"].items():
            print(f"Site {site}:")
            for param, value in site_data.items():
                print(f"  {param}: {value}")
        print()

# 按照位点和参数分组输出
def print_grouped_parameters_by_site(parameter_name, results):
    print(f"===== {parameter_name} =====")
    
    for site in [3,11,15,19,23]:  # 遍历所有位点
        print(f"\nSite {site}:")
        for folder in results:
            if site in results[folder]["sites"]:
                value = results[folder]["sites"][site].get(parameter_name, "N/A")
                print(f"{folder} - Site {site}: {value}")
    print()
def print_grouped_parameters_by_folder(parameter_name, results, overall=False):
    print(f"===== {parameter_name} =====")
    
    for folder in results:
        if overall:
            # 输出整体参数
            value = results[folder]["overall"].get(parameter_name, "N/A")
            print(f"{folder} - Overall: {value}")
        else:
            # 输出各个位点参数
            for site in results[folder]["sites"]:
                value = results[folder]["sites"][site].get(parameter_name, "N/A")
                print(f"{folder} - Site {site}: {value}")
    print()

# 运行输出
print_folder_based_output(results)

# 打印每个参数的值（可以按需调用）
print_grouped_parameters_by_site("d_band_center_site_up", results)
print_grouped_parameters_by_site("d_band_center_site_down", results)
print_grouped_parameters_by_site("d_band_filling_site_up", results)
print_grouped_parameters_by_site("d_band_filling_site_down", results)
print_grouped_parameters_by_site("d_band_skewness_site_up", results)
print_grouped_parameters_by_site("d_band_skewness_site_down", results)
print_grouped_parameters_by_site("d_band_kurtosis_site_up", results)
print_grouped_parameters_by_site("d_band_kurtosis_site_down", results)
print_grouped_parameters_by_site("d_band_width_site_up", results)
print_grouped_parameters_by_site("d_band_width_site_down", results)
print_grouped_parameters_by_site("d_band_unoccpy_site_up", results)
print_grouped_parameters_by_site("d_band_unoccpy_site_down", results)

# # 输出每个文件夹整体的值
# print_grouped_parameters_by_folder("d_band_center_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_center_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_filling_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_filling_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_skewness_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_skewness_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_kurtosis_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_kurtosis_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_width_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_width_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_unoccpy_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_unoccpy_down", results, overall=True)

Mn
Cr
Cu
Ni
Ti
Zn
Co
V
Fe
===== MnO2 =====
Overall Values:

Site-specific Values:
Site 3:
  d_band_center_site_up: -3.0554772086835653
  d_band_center_site_down: 2.098089349325176
  d_band_filling_site_up: 0.8279391838525695
  d_band_filling_site_down: 0.1536401555327098
  d_band_skewness_site_up: -0.21145350641754743
  d_band_skewness_site_down: -2.595994086451166
  d_band_kurtosis_site_up: 8.22554093693192
  d_band_kurtosis_site_down: 13.85043682672547
  d_band_width_site_up: 3.294558821954144
  d_band_width_site_down: 3.4288656672477678
  d_band_unoccpy_site_up: 1.943428174388375
  d_band_unoccpy_site_down: 3.3354516328177994
Site 11:
  d_band_center_site_up: -3.0602028611227237
  d_band_center_site_down: 2.1068893886480162
  d_band_filling_site_up: 0.8291309704074247
  d_band_filling_site_down: 0.1528059007554196
  d_band_skewness_site_up: -0.2066335566117403
  d_band_skewness_site_down: -2.6001348570123497
  d_band_kurtosis_site_up: 8.246204824853066
  d_band_kurtosis_site_down: 1

## 批量处理简洁版-按照位点更新

In [4]:
import pip
import pymatgen
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar
from pymatgen.io.cif import CifWriter
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import Slab, SlabGenerator, generate_all_slabs, Structure, Lattice, ReconstructionGenerator
import os
# pip.main(['install','ase'])
from ase import Atoms
from ase.io import read, write
from ase.visualize import view
from ase.build import surface
import numpy as np
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType
# 文件夹路径和参数初始化
folder_path = "rest-MO2/3d/slab/clean/DOS/"
all_folders = os.listdir(folder_path)

site_information = {
    "TiO2":(15,23),
    "VO2":(15,23),
    "CrO2":(11,19),                               
    "MnO2":(15,23),
    "FeO2":(15,23),
    "CoO2":(3,11),
    "NiO2":(11,19),
    "CuO2":(3,11),
    "ScO2":(7,15),
    "ZnO2":(3,11)
}
# site_information = {
#     # "HfO2":(7,15),
#     # "TaO2":(11,19),
#     # "WO2":(3,11),                               
#     # "ReO2":(15,23),
#     # "OsO2":(7,15),
#     # "IrO2":(15,23),
#     # "PtO2":(7,15),
#     # "AuO2":(7,15),
#     "CoO2":(3,11)
#     # "ScO2":(7,15),
# }
# site_information = {
#     "AgO2":(7,15),
#     "CdO2":(7,15),
#     "MoO2":(3,11),                               
#     "NbO2":(7,15),
#     "PdO2":(7,15),
#     "RhO2":(7,15),
#     "RuO2":(7,15),
#     "TcO2":(7,15),
#     "ZrO2":(11,19)
# }
# 用于存储结果的字典，按文件夹和参数存储
results = {folder: {"sites": {}} for folder in all_folders}

# 处理每个文件夹的 dos 数据
for folder in all_folders:
    data_path_vasprun = f"{folder_path}/{folder}/vasprun.xml"
    vasprun = Vasprun(data_path_vasprun)
    complete_dos = vasprun.complete_dos  # 获取完整的 DOS 类
    A = folder[:-2]
    elements_from_bulk = folder[:-2] # 元素
    sites_to_analyze = site_information.get(folder, [])
    print(f"{elements_from_bulk}")
    # 处理每个位点的数据
    for site in sites_to_analyze:
        if site not in results[folder]["sites"]:
            results[folder]["sites"][site] = {}
        results[folder]["sites"][site]["d_band_center_site"] = (
            str(complete_dos.get_band_center(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[-20, 20],
                band=OrbitalType.d
            )),
            str(complete_dos.get_band_center(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[-20, 20],
                band=OrbitalType.d
            ))
        )
        results[folder]["sites"][site]["d_band_filling_site"] = (
            str(complete_dos.get_band_filling(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                band=OrbitalType.d
            )),
            str(complete_dos.get_band_filling(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                band=OrbitalType.d
            ))
        )
        results[folder]["sites"][site]["d_band_skewness_site"] = (
            str(complete_dos.get_band_skewness(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[-20, 20],
                band=OrbitalType.d
            )),
            str(complete_dos.get_band_skewness(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[-20, 20],
                band=OrbitalType.d
            ))
        )
        results[folder]["sites"][site]["d_band_kurtosis_site"] = (
            str(complete_dos.get_band_kurtosis(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[-20, 20],
                band=OrbitalType.d
            )),
            str(complete_dos.get_band_kurtosis(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[-20, 20],
                band=OrbitalType.d
            ))
        )
        results[folder]["sites"][site]["d_band_width_site"] = (
            str(complete_dos.get_band_width(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[-20, 20],
                band=OrbitalType.d
            )),
            str(complete_dos.get_band_width(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[-20, 20],
                band=OrbitalType.d
            ))
        )
        results[folder]["sites"][site]["d_band_unoccpy_site"] = (
            str(complete_dos.get_band_center(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[0, 20],
                band=OrbitalType.d
            )),
            str(complete_dos.get_band_center(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[0, 20],
                band=OrbitalType.d
            ))
        )
# 按照位点和参数分组输出
def print_grouped_parameters_by_folder(parameter_name, results, site_information):
    print(f"===== {parameter_name} =====")
    site_groups = {}  # 存储不同 (site1, site2) 组合的文件夹
    
    for folder in results:
        site_tuple = site_information.get(folder, None)  # 取出该文件夹的位点元组
        if site_tuple is None:
            continue
        
        if site_tuple not in site_groups:
            site_groups[site_tuple] = []  # 初始化该组合的列表
        
        site_values = []
        for site in site_tuple:  # 遍历该组合的两个位点
            value = results[folder]["sites"].get(site, {}).get(parameter_name, "N/A")
            if isinstance(value, tuple):
                site_values.append(", ".join(value))  # 输出格式为 `value1, value2`
            else:
                site_values.append(value)
        
        site_groups[site_tuple].append(f"{folder}: {' | '.join(site_values)}")
    
    # 按照不同位点组合输出
    for site_tuple, folders in site_groups.items():
        print(f"\nSites {site_tuple}:")
        for entry in folders:
            print(entry)
    print()
# 打印每个参数的值（可以按需调用）
print_grouped_parameters_by_folder("d_band_center_site", results, site_information)
print_grouped_parameters_by_folder("d_band_filling_site", results, site_information)
print_grouped_parameters_by_folder("d_band_skewness_site", results, site_information)
print_grouped_parameters_by_folder("d_band_kurtosis_site", results, site_information)
print_grouped_parameters_by_folder("d_band_width_site", results, site_information)
print_grouped_parameters_by_folder("d_band_unoccpy_site", results, site_information)


# # 输出每个文件夹整体的值
# print_grouped_parameters_by_folder("d_band_center_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_center_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_filling_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_filling_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_skewness_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_skewness_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_kurtosis_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_kurtosis_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_width_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_width_down", results, overall=True)
# print_grouped_parameters_by_folder("d_band_unoccpy_up", results, overall=True)
# print_grouped_parameters_by_folder("d_band_unoccpy_down", results, overall=True)
# # # 按照文件夹进行输出
# def print_folder_based_output(results):
#     for folder, data in results.items():
#         print(f"===== {folder} =====")
        
#         # 打印整体数据
#         print("Overall Values:")
#         for param, value in data["overall"].items():
#             print(f"{param}: {value}")
        
#         # 打印每个位点的数据
#         print("\nSite-specific Values:")
#         for site, site_data in data["sites"].items():
#             print(f"Site {site}:")
#             for param, value in site_data.items():
#                 print(f"  {param}: {value}")
#         print()
# def print_grouped_parameters_by_folder(parameter_name, results, overall=False):
#     print(f"===== {parameter_name} =====")
    
#     for folder in results:
#         if overall:
#             # 输出整体参数
#             value = results[folder]["overall"].get(parameter_name, "N/A")
#             print(f"{folder} - Overall: {value}")
#         else:
#             # 输出各个位点参数
#             for site in results[folder]["sites"]:
#                 value = results[folder]["sites"][site].get(parameter_name, "N/A")
#                 print(f"{folder} - Site {site}: {value}")
#     print()

# 运行输出
# print_folder_based_output(results)

Mn
Cr
Cu
Ni
Ti
Zn
Co
V
Fe
Sc
===== d_band_center_site =====

Sites (15, 23):
MnO2: -2.7589057204392287, 1.877125338244218 | -2.749200284170435, 1.8731236619535283
TiO2: 1.2678562660716228, 1.2482053735181564 | 1.2688849988652473, 1.256285103113904
VO2: -0.9127474705424673, 0.7791675051168544 | -0.938627018929197, 0.7945446659675257
FeO2: -5.52702798885482, 0.8891451613992434 | -5.528540882319626, 0.887120746248176

Sites (11, 19):
CrO2: -1.173631122432493, 1.9697746951217239 | -1.5086941705889374, 0.4127666757032812
NiO2: -2.997180300784657, -3.6949284597288408 | -2.9671279335812004, -3.6229922985408014

Sites (3, 11):
CuO2: -3.6320069724911765, -3.4764365044304797 | -3.5791266807097477, -3.494491289999741
ZnO2: -6.350497700090274, -5.973622900121506 | -6.357681151977782, -5.966508674556145
CoO2: -7.654438503926223, 0.06595087791305018 | -7.1371691895058245, -0.1826641935882826

Sites (7, 15):
ScO2: 4.194510575556166, 4.113226859864967 | 4.220788803777146, 4.159420694970348

===== d_ba

## 读取金属的p带数据-总DOS

In [ ]:
site_information = {
    "TiO2":(51),
    "VO2":(51),
    "CrO2":(43),                               
    "MnO2":(51),
    "FeO2":(51),
    "CoO2":(43),
    "NiO2":(43),
    "CuO2":(43),
    "ZnO2":(43)
}

## Pdos批量计算-简洁版

In [5]:
import pip
import pymatgen
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar
from pymatgen.io.cif import CifWriter
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import Slab, SlabGenerator, generate_all_slabs, Structure, Lattice, ReconstructionGenerator
import os
# pip.main(['install','ase'])
from ase import Atoms
from ase.io import read, write
from ase.visualize import view
from ase.build import surface
import numpy as np
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType
# 文件夹路径和参数初始化
folder_path = "rest-MO2/3d/slab/clean/DOS/"
all_folders = os.listdir(folder_path)

site_information = {
    # "TiO2":(51),
    # "VO2":(51),
    # "CrO2":(43),                               
    # "MnO2":(51),
    # "FeO2":(51),
    # "CoO2":(43),
    # "NiO2":(43),
    # "CuO2":(43),
    # "ZnO2":(43),
    "CoO2":(43)
}
# site_information = {
#     "HfO2":(51),
#     "TaO2":(51),
#     "WO2":(43),                               
#     "ReO2":(67),
#     "OsO2":(51),
#     "IrO2":(51),
#     "PtO2":(51),
#     "AuO2":(51)
# }
# site_information = {
#     "ZrO2":(43),
#     "NbO2":(51),
#     "MoO2":(51),                               
#     "TcO2":(43),
#     "RuO2":(51),
#     "RhO2":(51),
#     "PdO2":(51),
#     "AgO2":(51),
#     "CdO2":(51)
# }
# 用于存储结果的字典，按文件夹和参数存储
results = {folder: {"sites": {}} for folder in all_folders}

# 处理每个文件夹的 dos 数据
for folder in all_folders:
    data_path_vasprun = f"{folder_path}/{folder}/vasprun.xml"
    vasprun = Vasprun(data_path_vasprun, parse_dos=True)
    complete_dos = vasprun.complete_dos  # 获取完整的 DOS 类
    # A = folder[:-2]
    elements_from_bulk = {"O"} # 元素
    sites_to_analyze = site_information.get(folder, [])
    if isinstance(sites_to_analyze, int):  # Ensure it's iterable
        sites_to_analyze = [sites_to_analyze]
    print(f"{elements_from_bulk}")
    # 处理每个位点的数据
    for site in sites_to_analyze:
        if site not in results[folder]["sites"]:
            results[folder]["sites"][site] = {}
        results[folder]["sites"][site]["p_band_center_site"] = (
            str(complete_dos.get_band_center(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[-20, 20],
                band=OrbitalType.p
            )),
            str(complete_dos.get_band_center(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[-20, 20],
                band=OrbitalType.p
            ))
        )
        results[folder]["sites"][site]["p_band_filling_site"] = (
            str(complete_dos.get_band_filling(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                band=OrbitalType.p
            )),
            str(complete_dos.get_band_filling(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                band=OrbitalType.p
            ))
        )
        results[folder]["sites"][site]["p_band_skewness_site"] = (
            str(complete_dos.get_band_skewness(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[-20, 20],
                band=OrbitalType.p
            )),
            str(complete_dos.get_band_skewness(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[-20, 20],
                band=OrbitalType.p
            ))
        )
        results[folder]["sites"][site]["p_band_kurtosis_site"] = (
            str(complete_dos.get_band_kurtosis(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[-20, 20],
                band=OrbitalType.p
            )),
            str(complete_dos.get_band_kurtosis(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[-20, 20],
                band=OrbitalType.p
            ))
        )
        results[folder]["sites"][site]["p_band_width_site"] = (
            str(complete_dos.get_band_width(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[-20, 20],
                band=OrbitalType.p
            )),
            str(complete_dos.get_band_width(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[-20, 20],
                band=OrbitalType.p
            ))
        )
        results[folder]["sites"][site]["p_band_unoccpy_site"] = (
            str(complete_dos.get_band_center(
                sites=[complete_dos.structure[site]],
                spin=Spin.up,
                erange=[0, 20],
                band=OrbitalType.p
            )),
            str(complete_dos.get_band_center(
                sites=[complete_dos.structure[site]],
                spin=Spin.down,
                erange=[0, 20],
                band=OrbitalType.p
            ))
        )
# 按照位点和参数分组输出
def print_grouped_parameters_by_folder(parameter_name, results, site_information):
    print(f"===== {parameter_name} =====")
    site_groups = {}  # 存储不同 (site1, site2) 组合的文件夹
    
    for folder in results:
        site_tuple = site_information.get(folder, None)  # 取出该文件夹的位点元组
        if isinstance(site_tuple, int):  
            site_tuple = (site_tuple,)  # Convert to a tuple if it's a single integer
        if site_tuple is None:
            continue
        
        if site_tuple not in site_groups:
            site_groups[site_tuple] = []  # 初始化该组合的列表
        
        site_values = []
        for site in site_tuple:  # 遍历该组合的两个位点
            value = results[folder]["sites"].get(site, {}).get(parameter_name, "N/A")
            if isinstance(value, tuple):
                site_values.append(", ".join(value))  # 输出格式为 `value1, value2`
            else:
                site_values.append(value)
        
        site_groups[site_tuple].append(f"{folder}: {' | '.join(site_values)}")
    
    # 按照不同位点组合输出
    for site_tuple, folders in site_groups.items():
        print(f"\nSites {site_tuple}:")
        for entry in folders:
            print(entry)
    print()
# 打印每个参数的值（可以按需调用）
print_grouped_parameters_by_folder("p_band_center_site", results, site_information)
print_grouped_parameters_by_folder("p_band_filling_site", results, site_information)
print_grouped_parameters_by_folder("p_band_skewness_site", results, site_information)
print_grouped_parameters_by_folder("p_band_kurtosis_site", results, site_information)
print_grouped_parameters_by_folder("p_band_width_site", results, site_information)
print_grouped_parameters_by_folder("p_band_unoccpy_site", results, site_information)

{'O'}
{'O'}
{'O'}
{'O'}
{'O'}
{'O'}
{'O'}
{'O'}
{'O'}
{'O'}
===== p_band_center_site =====

Sites (43,):
CoO2: -1.7831454154247492, -1.0232282086286586

===== p_band_filling_site =====

Sites (43,):
CoO2: 0.9445876657943464, 0.6857764759932842

===== p_band_skewness_site =====

Sites (43,):
CoO2: 0.19787536406037928, 0.502880829869156

===== p_band_kurtosis_site =====

Sites (43,):
CoO2: 12.703353743792718, 8.532086741736647

===== p_band_width_site =====

Sites (43,):
CoO2: 2.780152833085696, 2.943694130110827

===== p_band_unoccpy_site =====

Sites (43,):
CoO2: 4.642137064660079, 2.2813408898938934



In [8]:
import pip
import pymatgen
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar
from pymatgen.io.cif import CifWriter
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.core.surface import Slab, SlabGenerator, generate_all_slabs, Structure, Lattice, ReconstructionGenerator
import os
# pip.main(['install','ase'])
from ase import Atoms
from ase.io import read, write
from ase.visualize import view
from ase.build import surface
import numpy as np
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType

# 文件夹路径和参数初始化
folder_path = "rest-MO2/3d/slab/clean/DOS"
all_folders = os.listdir(folder_path)

# 用于存储结果的字典，按文件夹和参数存储
results = {folder: {"overall": {}, "sites": {}} for folder in all_folders}
# 处理每个文件夹的 dos 数据
for folder in all_folders:
    data_path_vasprun = f"{folder_path}/{folder}/vasprun.xml"
##    POS = read(f"{folder_path}/{folder}/POSCAR")
##    view(POS)
    vasprun = Vasprun(data_path_vasprun)
    complete_dos = vasprun.complete_dos  # 获取完整的 DOS 类
##    A = folder[:2]
    elements_from_bulk = {"O"}  # 元素
    sites_to_analyze = [51]  # 位点

    # 整体值计算
    results[folder]["overall"]["p_band_center_up"] = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.up, erange=[-20, 20], band=OrbitalType.p)
    results[folder]["overall"]["p_band_center_down"] = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.down, erange=[-20, 20], band=OrbitalType.p)
    results[folder]["overall"]["p_band_filling_up"] = complete_dos.get_band_filling(elements=elements_from_bulk, spin=Spin.up, band=OrbitalType.p)
    results[folder]["overall"]["p_band_filling_down"] = complete_dos.get_band_filling(elements=elements_from_bulk, spin=Spin.down, band=OrbitalType.p)
    results[folder]["overall"]["p_band_skewness_up"] = complete_dos.get_band_skewness(elements=elements_from_bulk, spin=Spin.up, erange=[-20, 20], band=OrbitalType.p)
    results[folder]["overall"]["p_band_skewness_down"] = complete_dos.get_band_skewness(elements=elements_from_bulk, spin=Spin.down, erange=[-20, 20], band=OrbitalType.p)
    results[folder]["overall"]["p_band_kurtosis_up"] = complete_dos.get_band_kurtosis(elements=elements_from_bulk, spin=Spin.up, erange=[-20, 20], band=OrbitalType.p)
    results[folder]["overall"]["p_band_kurtosis_down"] = complete_dos.get_band_kurtosis(elements=elements_from_bulk, spin=Spin.down, erange=[-20, 20], band=OrbitalType.p)
    results[folder]["overall"]["p_band_width_up"] = complete_dos.get_band_width(elements=elements_from_bulk, spin=Spin.up, erange=[-20, 0], band=OrbitalType.p)
    results[folder]["overall"]["p_band_width_down"] = complete_dos.get_band_width(elements=elements_from_bulk, spin=Spin.down, erange=[-20, 0], band=OrbitalType.p)
    results[folder]["overall"]["p_band_unoccpy_up"] = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.up, erange=[0, 20], band=OrbitalType.p)
    results[folder]["overall"]["p_band_unoccpy_down"] = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.down, erange=[0, 20], band=OrbitalType.p)
    # 处理每个位点的数据
    for site in sites_to_analyze:
        if site not in results[folder]["sites"]:
            results[folder]["sites"][site] = {}

        
        # 每个位点的计算
        results[folder]["sites"][site]["p_band_center_site_up"] = complete_dos.get_band_center(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[-20, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_center_site_down"] = complete_dos.get_band_center(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[-20, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_filling_site_up"] = complete_dos.get_band_filling(sites=[complete_dos.structure[site]], spin=Spin.up, band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_filling_site_down"] = complete_dos.get_band_filling(sites=[complete_dos.structure[site]], spin=Spin.down, band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_skewness_site_up"] = complete_dos.get_band_skewness(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[-20, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_skewness_site_down"] = complete_dos.get_band_skewness(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[-20, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_kurtosis_site_up"] = complete_dos.get_band_kurtosis(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[-20, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_kurtosis_site_down"] = complete_dos.get_band_kurtosis(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[-20, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_width_site_up"] = complete_dos.get_band_width(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[-20, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_width_site_down"] = complete_dos.get_band_width(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[-20, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_unoccpy_site_up"] = complete_dos.get_band_center(sites=[complete_dos.structure[site]], spin=Spin.up, erange=[0, 20], band=OrbitalType.p)
        results[folder]["sites"][site]["p_band_unoccpy_site_down"] = complete_dos.get_band_center(sites=[complete_dos.structure[site]], spin=Spin.down, erange=[0, 20], band=OrbitalType.p)        
# 按照文件夹进行输出
def print_folder_based_output(results):
    for folder, data in results.items():
        print(f"===== {folder} =====")
        
        # 打印整体数据
        print("Overall Values:")
        for param, value in data["overall"].items():
            print(f"{param}: {value}")
        
        # 打印每个位点的数据
        print("\nSite-specific Values:")
        for site, site_data in data["sites"].items():
            print(f"Site {site}:")
            for param, value in site_data.items():
                print(f"{param}: {value}")
        print()
        
# 按照位点和参数分组输出
def print_grouped_parameters_by_site(parameter_name, results):
    print(f"===== {parameter_name} =====")
    
    for site in [51]:  # 遍历所有位点
        print(f"\nSite {site}:")
        for folder in results:
            if site in results[folder]["sites"]:
                value = results[folder]["sites"][site].get(parameter_name, "N/A")
                print(f"{folder} - Site {site}: {value}")
    print()
def print_grouped_parameters_by_folder(parameter_name, results, overall=False):
    print(f"===== {parameter_name} =====")
    
    for folder in results:
        if overall == True :
            # 输出整体参数
            value = results[folder]["overall"].get(parameter_name, "N/A")
            print(f"{folder} - Overall: {value}")
        else:
            # 输出各个位点参数
            for site in results[folder]["sites"]:
                value = results[folder]["sites"][site].get(parameter_name, "N/A")
                print(f"{folder} - Site {site}: {value}")
    print()

# 运行输出
print_folder_based_output(results)

# 打印每个参数的值（可以按需调用）
print_grouped_parameters_by_site("p_band_center_site_up", results)
print_grouped_parameters_by_site("p_band_center_site_down", results)
print_grouped_parameters_by_site("p_band_filling_site_up", results)
print_grouped_parameters_by_site("p_band_filling_site_down", results)
print_grouped_parameters_by_site("p_band_skewness_site_up", results)
print_grouped_parameters_by_site("p_band_skewness_site_down", results)
print_grouped_parameters_by_site("p_band_kurtosis_site_up", results)
print_grouped_parameters_by_site("p_band_kurtosis_site_down", results)
print_grouped_parameters_by_site("p_band_width_site_up", results)
print_grouped_parameters_by_site("p_band_width_site_down", results)
print_grouped_parameters_by_site("p_band_unoccpy_site_up", results)
print_grouped_parameters_by_site("p_band_unoccpy_site_down", results)

# 输出每个文件夹整体的值
print_grouped_parameters_by_folder("p_band_center_up", results, overall=True)
print_grouped_parameters_by_folder("p_band_center_down", results, overall=True)
print_grouped_parameters_by_folder("p_band_filling_up", results, overall=True)
print_grouped_parameters_by_folder("p_band_filling_down", results, overall=True)
print_grouped_parameters_by_folder("p_band_skewness_up", results, overall=True)
print_grouped_parameters_by_folder("p_band_skewness_down", results, overall=True)
print_grouped_parameters_by_folder("p_band_kurtosis_up", results, overall=True)
print_grouped_parameters_by_folder("p_band_kurtosis_down", results, overall=True)
print_grouped_parameters_by_folder("p_band_width_up", results, overall=True)
print_grouped_parameters_by_folder("p_band_width_down", results, overall=True)
print_grouped_parameters_by_folder("p_band_unoccpy_up", results, overall=True)
print_grouped_parameters_by_folder("p_band_unoccpy_down", results, overall=True)

/data2/home/luodh/anaconda3/envs/ads/lib/python3.9/site-packages/pymatgen/io/vasp/outputs.py:350: UnconvergedVASPWarning: rest-MO2/3d/slab/clean/DOS/NiO2/vasprun.xml is an unconverged VASP run.
Electronic convergence reached: False.
Ionic convergence reached: True.
  warnings.warn(msg, UnconvergedVASPWarning)


===== MnO2 =====
Overall Values:
p_band_center_up: -1.7665971462069892
p_band_center_down: -1.8151230458896905
p_band_filling_up: 0.792891368554159
p_band_filling_down: 0.8288788889599743
p_band_skewness_up: 0.8845454087529014
p_band_skewness_down: 1.354597381477359
p_band_kurtosis_up: 5.761714972161896
p_band_kurtosis_down: 5.721358884045756
p_band_width_up: 2.190853510607648
p_band_width_down: 1.6260054604203857
p_band_unoccpy_up: 2.8830871322653144
p_band_unoccpy_down: 4.725758296021049

Site-specific Values:
Site 51:
p_band_center_site_up: -0.9781181132086364
p_band_center_site_down: -1.3360052151435207
p_band_filling_site_up: 0.7268853496185088
p_band_filling_site_down: 0.829892752376955
p_band_skewness_site_up: 0.4161774318623913
p_band_skewness_site_down: 0.8882301113512251
p_band_kurtosis_site_up: 8.108000331756557
p_band_kurtosis_site_down: 8.007095990382972
p_band_width_site_up: 3.030543545183233
p_band_width_site_down: 3.2939200062893907
p_band_unoccpy_site_up: 2.34678516568

In [9]:
import pymatgen 
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType
import ase
from ase import Atoms
from ase.io import read, write
from ase.visualize import view
from ase.build import surface
floder_path = "rest-MO2/doped/other-TiO2/dos/Cr/" #输入读取的dos数据路径
data_path_vasprun = f"{floder_path}/vasprun.xml" #要读取的文件，pymatgen推荐读取vasprun.xml
POS = read(f"{floder_path}/POSCAR")
view(POS)
vasprun = Vasprun(data_path_vasprun) #读取vasprun.xml的文件信息
efermi = vasprun.efermi
orbit_d=OrbitalType.
complete_dos = vasprun.complete_dos  #得到Complete_dos类
structure = complete_dos.structure  #得到结构信息
print(structure) #打印类似POSCAR的结构信息
sites = [0,8] #选取需要读取的原子序号
for i in sites:
    d_b_c_up = complete_dos.get_band_center( sites=[structure[i]], spin=Spin.up, erange=[-20,20], band=orbit_d)
    d_b_c_down = complete_dos.get_band_center( sites=[structure[i]], spin=Spin.down, erange=[-20,20], band=orbit_d)
    d_b_c_up_unoccupy = complete_dos.get_band_center(sites=[structure[i]], spin=Spin.up, erange=[0,20], band=orbit_d)
    d_b_c_down_unoccupy = complete_dos.get_band_center(sites=[structure[i]], spin=Spin.down, erange=[0,20], band=orbit_d)
    d_b_f_up = complete_dos.get_band_filling( sites=[structure[i]], spin=Spin.up, band=orbit_d)
    d_b_f_down = complete_dos.get_band_filling( sites=[structure[i]], spin=Spin.down, band=orbit_d)
    d_s_up = complete_dos.get_band_skewness( sites=[structure[i]], spin=Spin.up, erange=[-20,20], band=orbit_d)
    d_s_down = complete_dos.get_band_skewness( sites=[structure[i]], spin=Spin.down, erange=[-20,20], band=orbit_d)
    d_k_up = complete_dos.get_band_kurtosis( sites=[structure[i]], spin=Spin.up, erange=[-20,20], band=orbit_d)
    d_k_down = complete_dos.get_band_kurtosis( sites=[structure[i]], spin=Spin.down, erange=[-20,20], band=orbit_d)
    d_b_w_up = complete_dos.get_band_width( sites=[structure[i]], spin=Spin.up, erange=[-20,0], band=orbit_d)
    d_b_w_fown = complete_dos.get_band_width( sites=[structure[i]], spin=Spin.down, erange=[-20,0], band=orbit_d)
    print(f'位点{i}自旋向下的dband-center:',f'位点{i}自旋向下的dband-center:',)
    print(f'位点{i}自旋向下的d_un_band-center:',f'位点{i}自旋向下的d_un_band-center:',)
    print(f'位点{i}的自旋向上的d带填充:',f'位点{i}位点1的自旋向下的d带填充:')
    print(f'位点{i}自旋向上的偏度:',f'位点{i}自旋向下的偏度')
    print(f'位点{i}自旋向上的峰度:',f'位点{i}自旋向下的峰度:')
    print(f'位点{i}的自旋向上的d带宽度:',f'位点{i}的自旋向下的d带宽度:')
    print(d_b_c_up,d_b_c_down)
    print(d_b_c_up_unoccupy,d_b_c_down_unoccupy)
    print(d_b_f_up,d_b_f_down)
    print(d_k_up,d_k_down)
    print(d_s_up,d_s_down)
    print(d_b_w_up,d_b_w_fown)

Full Formula (Ti23 Cr1 O48)
Reduced Formula: Ti23CrO48
abc   :   9.069400   6.603700  27.478700
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (72)
  #  SP           a         b         c  selective_dynamics
---  ----  --------  --------  --------  ---------------------
  0  Cr    0.633342  0.59994   0.560447  [True, True, True]
  1  Ti    0.13333   0.6       0.19239   [False, False, False]
  2  Ti    0.13333   0.1       0.31255   [False, False, False]
  3  Ti    0.133343  0.600054  0.436282  [True, True, True]
  4  Ti    0.133348  0.100026  0.548381  [True, True, True]
  5  Ti    0.3       0.1       0.19239   [False, False, False]
  6  Ti    0.3       0.6       0.31255   [False, False, False]
  7  Ti    0.300298  0.10004   0.431048  [True, True, True]
  8  Ti    0.300315  0.599992  0.559471  [True, True, True]
  9  Ti    0.46667   0.6       0.19239   [False, False, False]
 10  Ti    0.46667   0.1       0.31255   [False, False, False]
 11  Ti   

## 获取某元素的总p-DOS数据

In [7]:
import pymatgen 
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType
import ase
from ase import Atoms
from ase.io import read, write
from ase.visualize import view
from ase.build import surface
floder_path = "rest-MO2/IrO2/slab-211/dos/" #输入读取的dos数据路径
data_path_vasprun = f"{floder_path}/vasprun.xml" #要读取的文件，pymatgen推荐读取vasprun.xml
POS = read(f"{floder_path}/POSCAR")
view(POS)
vasprun = Vasprun(data_path_vasprun) #读取vasprun.xml的文件信息
efermi = vasprun.efermi
complete_dos = vasprun.complete_dos  #得到Complete_dos类
structure = complete_dos.structure  #得到结构信息
elements_from_bulk = "O"
i = 90   #选取需要读取的原子序号
p_b_c_up = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.up, erange=[-20,20], band=OrbitalType.p)
p_b_c_down = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.down, erange=[-20,20], band=OrbitalType.p)
p_b_c_up_unoccupy = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.up, erange=[0,20], band=OrbitalType.p)
p_b_c_down_unoccupy = complete_dos.get_band_center(elements=elements_from_bulk, spin=Spin.down, erange=[0,20], band=OrbitalType.p)
p_b_f_up = complete_dos.get_band_filling(elements=elements_from_bulk, spin=Spin.up, band=OrbitalType.p)
p_b_f_down = complete_dos.get_band_filling(elements=elements_from_bulk, spin=Spin.down, band=OrbitalType.p)
p_s_up = complete_dos.get_band_skewness(elements=elements_from_bulk, spin=Spin.up, erange=[-20,20], band=OrbitalType.p)
p_s_down = complete_dos.get_band_skewness(elements=elements_from_bulk, spin=Spin.down, erange=[-20,20], band=OrbitalType.p)
p_k_up = complete_dos.get_band_kurtosis(elements=elements_from_bulk, spin=Spin.up, erange=[-20,20], band=OrbitalType.p)
p_k_down = complete_dos.get_band_kurtosis(elements=elements_from_bulk, spin=Spin.down, erange=[-20,20], band=OrbitalType.p)
p_b_w_up = complete_dos.get_band_width(elements=elements_from_bulk, spin=Spin.up, erange=[-20,0], band=OrbitalType.p)
p_b_w_fown = complete_dos.get_band_width(elements=elements_from_bulk, spin=Spin.down, erange=[-20,0], band=OrbitalType.p)
print(structure) #打印类似POSCAR的结构信息
print(f'位点{elements_from_bulk}的自旋向下的pband-center:',p_b_c_up,f':位点{elements_from_bulk}的自旋向下的pband-center:',p_b_c_down,)
print(f'位点{elements_from_bulk}的自旋向下的p_un_band-center:',p_b_c_up_unoccupy,f':位点{elements_from_bulk}的自旋向下的p_un_band-center:',p_b_c_down_unoccupy,)
print(f'位点{elements_from_bulk}的自旋向上的峰度:',p_k_up,f':位点{elements_from_bulk}的自旋向下的峰度:',p_k_down)
print(f'位点{elements_from_bulk}的自旋向上的偏度:',p_s_up,f':位点{elements_from_bulk}的自旋向下的偏度:',p_s_down)
print(f'位点{elements_from_bulk}的自旋向上的p带填充:',p_b_f_up,f':位点{elements_from_bulk}的自旋向下的p带填充:',p_b_f_down)
print(f'位点{elements_from_bulk}的自旋向上的p带宽度:',p_b_w_up,f':位点{elements_from_bulk}的自旋向下的p带宽度:',p_b_w_fown)

ParseError: no element found: line 61310, column 0 (<string>)

usage: ase [-h] [--version] [-T]
           {help,info,test,gui,db,run,band-structure,build,dimensionality,eos,ulm,find,nebplot,nomad-upload,nomad-get,convert,reciprocal,completion,diff,exec}
           ...
ase: error: TclError: couldn't connect to display "localhost:12.0"
To get a full traceback, use: ase -T gui ...


## 获取某位点的p带信息

In [5]:
import pymatgen 
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType
import ase
from ase import Atoms
from ase.io import read, write
from ase.visualize import view
from ase.build import surface
floder_path = "/data2/home/luodh/VOx/VO2/slab/end2/5layer/Ov-magmom/reaction/dos/band/clean" #输入读取的dos数据路径
data_path_vasprun = f"{floder_path}/vasprun.xml" #要读取的文件，pymatgen推荐读取vasprun.xml
POS = read(f"{floder_path}/POSCAR")
view(POS)
vasprun = Vasprun(data_path_vasprun) #读取vasprun.xml的文件信息
efermi = vasprun.efermi
complete_dos = vasprun.complete_dos  #得到Complete_dos类
structure = complete_dos.structure  #得到结构信息
#print(structure) #打印类似POSCAR的结构信息
sites = [14,34] #选取需要读取的原子序号
for i in sites:
    p_b_c_up = complete_dos.get_band_center( sites=[structure[i]], spin=Spin.up, erange=[-20,20], band=OrbitalType.p)
    p_b_c_down = complete_dos.get_band_center( sites=[structure[i]], spin=Spin.down, erange=[-20,20], band=OrbitalType.p)
    p_b_c_up_unoccupy = complete_dos.get_band_center(sites=[structure[i]], spin=Spin.up, erange=[0,20], band=OrbitalType.p)
    p_b_c_down_unoccupy = complete_dos.get_band_center(sites=[structure[i]], spin=Spin.down, erange=[0,20], band=OrbitalType.p)
    p_b_f_up = complete_dos.get_band_filling( sites=[structure[i]], spin=Spin.up, band=OrbitalType.p)
    p_b_f_down = complete_dos.get_band_filling( sites=[structure[i]], spin=Spin.down, band=OrbitalType.p)
    p_s_up = complete_dos.get_band_skewness( sites=[structure[i]], spin=Spin.up, erange=[-20,20], band=OrbitalType.p)
    p_s_down = complete_dos.get_band_skewness( sites=[structure[i]], spin=Spin.down, erange=[-20,20], band=OrbitalType.p)
    p_k_up = complete_dos.get_band_kurtosis( sites=[structure[i]], spin=Spin.up, erange=[-20,20], band=OrbitalType.p)
    p_k_down = complete_dos.get_band_kurtosis( sites=[structure[i]], spin=Spin.down, erange=[-20,20], band=OrbitalType.p)
    p_b_w_up = complete_dos.get_band_width( sites=[structure[i]], spin=Spin.up, erange=[-20,0], band=OrbitalType.p)
    p_b_w_fown = complete_dos.get_band_width( sites=[structure[i]], spin=Spin.down, erange=[-20,0], band=OrbitalType.p)
    print(f'位点{i}的自旋向下的pband-center:',p_b_c_up,f'位点{i}的自旋向下的pband-center:')
    print(f'位点{i}的自旋向下的p_un_band-center:',f'位点{i}的自旋向下的p_un_band-center:')
    print(f'位点{i}的自旋向上的p带填充:',f'位点{i}的自旋向下的p带填充:')
    print(f'位点{i}的自旋向上的偏度:',f'位点{i}的自旋向下的偏度')
    print(f'位点{i}的自旋向上的峰度:',f'位点{i}的自旋向下的峰度:')
    print(f'位点{i}的自旋向上的p带宽度:',f'位点{i}的自旋向下的p带宽度:')
    print(p_b_c_up,p_b_c_down)
    print(p_b_c_up_unoccupy,p_b_c_down_unoccupy)
    print(p_b_f_up,p_b_f_down)
    print(p_s_up,p_s_down)
    print(p_k_up,p_k_down)
    print(p_b_w_up,p_b_w_fown)

位点14的自旋向下的pband-center: -2.2631028888765217 位点14的自旋向下的pband-center:
位点14的自旋向下的p_un_band-center: 位点14的自旋向下的p_un_band-center:
位点14的自旋向上的p带填充: 位点14的自旋向下的p带填充:
位点14的自旋向上的偏度: 位点14的自旋向下的偏度
位点14的自旋向上的峰度: 位点14的自旋向下的峰度:
位点14的自旋向上的p带宽度: 位点14的自旋向下的p带宽度:
-2.2631028888765217 -2.44714484921609
2.1132842471487616 3.1592567651281125
0.8368772942574216 0.8901621253885681
-0.2776208923838114 0.3558838639043212
8.588320556492423 9.607447111547334
1.5936112115569219 1.4232078015918348
位点34的自旋向下的pband-center: -2.254470338257629 位点34的自旋向下的pband-center:
位点34的自旋向下的p_un_band-center: 位点34的自旋向下的p_un_band-center:
位点34的自旋向上的p带填充: 位点34的自旋向下的p带填充:
位点34的自旋向上的偏度: 位点34的自旋向下的偏度
位点34的自旋向上的峰度: 位点34的自旋向下的峰度:
位点34的自旋向上的p带宽度: 位点34的自旋向下的p带宽度:
-2.254470338257629 -2.4424891779729205
2.1202558541479064 3.162656516720486
0.8356291778815378 0.8900172681184403
-0.3035258510712696 0.3611363327439531
8.65678040117 9.575909846045446
1.6043646772831812 1.4219819578225423


In [21]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymatgen.io.vasp import Vasprun
from pymatgen.electronic_structure.core import Spin, OrbitalType

# ============ 用户输入参数 ============
folder_path = "rest-MO2/4d/slab/clean/DOS/MoO2"  # 指定文件夹路径（示例）
site_index = 7  # 指定需要提取的原子位点索引
save_csv = False  # 是否保存为 CSV

# ============ 读取 vasprun.xml ============
data_path_vasprun = os.path.join(folder_path, "vasprun.xml")
vasprun = Vasprun(data_path_vasprun)
complete_dos = vasprun.complete_dos
structure = complete_dos.structure  # 获取结构信息
efermi = vasprun.efermi  # 费米能级

# ============ 提取 PDOS 数据 ============
pdos = complete_dos.get_element_spd_dos(structure[site_index].species_string)
energies = complete_dos.energies - efermi  # 归一化能量轴（相对费米能级）

# ============ 提取 d 轨道 DOS ============
d_orbitals = OrbitalType.d
dos_data = {
    "Energy (eV)": energies
}
for spin in [Spin.up, Spin.down]:
    dos_data[f"d_total_{'up' if spin == Spin.up else 'down'}"] = sum(pdos[orb][spin] for orb in d_orbitals)

# ============ 计算带中心 ============
d_band_center_up = complete_dos.get_band_center([structure[site_index]], Spin.up, erange=[-20, 20], band=OrbitalType.d)
d_band_center_down = complete_dos.get_band_center([structure[site_index]], Spin.down, erange=[-20, 20], band=OrbitalType.d)

print(f"\n📌 位点 {site_index} 的 d 带中心 (eV)：")
print(f"   ⬆️ 自旋向上: {d_band_center_up:.3f}")
print(f"   ⬇️ 自旋向下: {d_band_center_down:.3f}")

# ============ 保存到 CSV ============
if save_csv:
    df = pd.DataFrame(dos_data)
    csv_filename = os.path.join(folder_path, f"DOS_site_{site_index}.csv")
    df.to_csv(csv_filename, index=False)
    print(f"\n✅ DOS 数据已保存至: {csv_filename}")

# ============ 绘制 DOS 图 ============
plt.figure(figsize=(6, 4))
plt.plot(energies, dos_data["d_total_up"], label="Spin Up", color="blue")
plt.plot(energies, -dos_data["d_total_down"], label="Spin Down", color="red")  # 反向绘制自旋向下
plt.axvline(0, color="black", linestyle="--", label="Fermi Level")
plt.xlabel("Energy (eV)")
plt.ylabel("DOS")
plt.legend()
plt.title(f"DOS for site {site_index}")
plt.tight_layout()
plt.savefig(os.path.join(folder_path, f"DOS_site_{site_index}.png"))
plt.show()

TypeError: 'OrbitalType' object is not iterable